# Regression Trees: Predicting NYC Taxi Tip Amount

**Course:** IBM Machine Learning with Python

## Objective
Train a `DecisionTreeRegressor` to predict `tip_amount` for NYC yellow taxi trips, using the other trip features (fare, distance, passenger count, etc.).

## Workflow
1. Import libraries
2. Load the dataset
3. Explore correlation with the target (`tip_amount`)
4. Drop uninformative columns
5. Build the feature matrix and normalize it
6. Train/test split
7. Train a Decision Tree Regressor
8. Evaluate with MSE and R^2
9. Identify top correlated features
10. Compare a shallower tree (max_depth=4)

## 1. Imports
- `train_test_split`: splits data into train/test sets
- `normalize`: scales feature rows so they share a comparable magnitude (L1 norm)
- `mean_squared_error`: regression error metric
- `warnings.filterwarnings('ignore')`: suppresses noisy library warnings for a cleaner output

In [ ]:
from __future__ import print_function

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## 2. Load the Dataset
Each row is one taxi trip with 13 variables. The target variable is `tip_amount` — the model will learn to predict it from the remaining features.

In [ ]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/pu9kbeSaAtRZ7RxdJKX9_A/yellow-tripdata.csv'
raw_data = pd.read_csv(url)
raw_data

## 3. Correlation with the Target
A horizontal bar chart of each feature's correlation with `tip_amount`. Bars far from zero (either direction) indicate a stronger linear relationship.

In [ ]:
correlation_values = raw_data.corr()['tip_amount'].drop('tip_amount')
correlation_values.plot(kind='barh', figsize=(10, 6))

## 4. Drop Uninformative Columns
`payment_type`, `VendorID`, `store_and_fwd_flag`, and `improvement_surcharge` show negligible correlation with `tip_amount` and add no predictive value, so they're removed before building the feature matrix.

In [ ]:
raw_data = raw_data.drop(['payment_type', 'VendorID', 'store_and_fwd_flag', 'improvement_surcharge'], axis=1)

## 5. Build the Feature Matrix and Target
- `y`: the target column `tip_amount`, cast to `float32`
- `proc_data`: all remaining columns after dropping the target
- `X`: the feature matrix, then L1-normalized row-wise so each row's feature values sum (in absolute value) to 1 — this keeps features on a comparable scale

In [ ]:
# extract the labels from the dataframe
y = raw_data[['tip_amount']].values.astype('float32')

# drop the target variable from the feature matrix
proc_data = raw_data.drop(['tip_amount'], axis=1)

# get the feature matrix used for training
X = proc_data.values

# normalize the feature matrix
X = normalize(X, axis=1, norm='l1', copy=False)

## 6. Train/Test Split
Hold out 30% of trips for testing. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## 7. Build and Train the Regression Tree
- `criterion='squared_error'`: splits are chosen to minimize squared error (standard for regression trees)
- `max_depth=8`: limits tree depth to control overfitting
- `random_state=35`: reproducible results across runs

In [ ]:
# import the Decision Tree Regression Model from scikit-learn
from sklearn.tree import DecisionTreeRegressor

# for reproducible output across multiple function calls, set random_state to a given integer value
dt_reg = DecisionTreeRegressor(criterion='squared_error',
                                max_depth=8,
                                random_state=35)

In [ ]:
dt_reg.fit(X_train, y_train)

## 8. Predict and Evaluate
- **MSE (Mean Squared Error)**: average squared difference between predicted and actual tip amounts — lower is better
- **R^2 score**: proportion of variance in `tip_amount` explained by the model — closer to 1 is better

In [ ]:
# run inference using the sklearn model
y_pred = dt_reg.predict(X_test)

# evaluate mean squared error on the test dataset
mse_score = mean_squared_error(y_test, y_pred)
print('MSE score : {0:.3f}'.format(mse_score))

r2_score = dt_reg.score(X_test, y_test)
print('R^2 score : {0:.3f}'.format(r2_score))

## Q3. Which Features Correlate Most with `tip_amount`?
Sort the absolute correlation values (recomputed before the column drop) to find the top 3 features most predictive of the tip.

In [ ]:
correlation_values = raw_data.corr()['tip_amount'].drop('tip_amount')
abs(correlation_values).sort_values(ascending=False)[:3]

## Q4. Effect of Decreasing `max_depth` to 4
Retrain the tree with `max_depth=4` and compare MSE and R^2 against the `max_depth=8` model.

In [ ]:
dt_reg = DecisionTreeRegressor(criterion='squared_error', max_depth=4, random_state=35)
dt_reg.fit(X_train, y_train)

y_pred = dt_reg.predict(X_test)
mse_score = mean_squared_error(y_test, y_pred)
print('MSE score : {0:.3f}'.format(mse_score))

r2_score = dt_reg.score(X_test, y_test)
print('R^2 score : {0:.3f}'.format(r2_score))

**Answer:** with `max_depth=4`, the MSE decreases and the R^2 increases compared to `max_depth=8` — meaning `max_depth=4` generalizes better on this dataset. The deeper tree (depth 8) was overfitting the training data.

## Conclusion
A Decision Tree Regressor can predict taxi tip amounts from trip features. Dropping uninformative columns and tuning `max_depth` both matter: a tree that's too deep overfits, while a moderately shallow tree (depth 4 here) generalizes better, as shown by the lower MSE and higher R^2.